In [ ]:
import os
import json
import time
import re
from typing import Dict
from datasets import load_dataset
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from huggingface_hub import login
from google.colab import userdata

# Initialize OpenAI client
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

def load_existing_ids(output_file: str) -> set:
    """Read JSONL and return set of existing 'id' values."""
    existing = set()
    if not os.path.exists(output_file):
        return existing

    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if "id" in obj:
                    existing.add(str(obj["id"]))
            except json.JSONDecodeError:
                continue
    return existing


def get_sample_id(sample, dataset_name: str, idx: int) -> str:
    """Stable id for skipping."""
    return str(sample.get("id", f"{dataset_name.lower()}_{idx}"))

def image_to_base64(image):
    """Convert PIL Image to base64 string"""
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode()

def identify_dataset_source(entry: Dict) -> str:
    """
    Identify which dataset an entry belongs to

    Args:
        entry: Dataset entry

    Returns:
        Dataset name or None
    """
    # Check if entry has 'id' field with dataset path
    entry_id = entry.get('id', '')

    # Handle image field - might be PIL Image or string
    image_path = entry.get('image', '')
    if hasattr(image_path, '__class__') and 'Image' in str(type(image_path)):
        # It's a PIL Image, can't search in it, use empty string
        image_path = ''
    else:
        image_path = str(image_path).lower()

    # Common patterns in LLaVA-CoT-o1-Instruct

    # IconQA: has 'iconqa'
    if 'iconqa' in entry_id.lower() or 'iconqa' in image_path.lower():
        return 'IconQA'

    return None

def translate_text(text: str, text_type: str = "question") -> str:
    """
    Translate text to Arabic

    Args:
        text: English text
        text_type: Type of text ("question" or "answer")

    Returns:
        Arabic translation
    """
    if text_type == "question":
        prompt = f"""ترجم السؤال التالي إلى اللغة العربية الفصحى:

{text}

قواعد:
- ترجم السؤال الكامل بما في ذلك أي مقدمات أو تعليمات
- استخدم لغة علمية دقيقة
- لا تضف أي تفسيرات إضافية

الترجمة:"""
    else:  # answer
        prompt = f"""ترجم الإجابة التالية إلى اللغة العربية الفصحى:

{text}

قواعد:
- ترجم الإجابة فقط دون إضافات
- استخدم لغة دقيقة

الترجمة:"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": "أنت مترجم محترف متخصص في ترجمة النصوص العلمية."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=500
        )
        translation = response.choices[0].message.content.strip()
        time.sleep(1)

        return translation

    except Exception as e:
        print(f"Translation error: {e}")
        return text

def build_content(base64_image, question, standard_answer):
    """Build the multimodal content for the chat completion."""
    content = []

    if base64_image is not None:
        content.append({
            "type": "image_url",
            "image_url": {"url": "data:image/jpeg;base64," + base64_image}
        })

    content.append({
        "type": "text",
        "text": (
            "لدي صورة وسؤال أريد منك الإجابة عليه. أحتاج منك اتباع التنسيق بدقة مع أربعة أقسام محددة: الملخص (SUMMARY)، الوصف (CAPTION)، التفكير (REASONING)، والخلاصة (CONCLUSION). "
            "من الضروري جداً الالتزام بهذا الهيكل تماماً وأن تتطابق الإجابة النهائية في الخلاصة مع الإجابة الصحيحة القياسية بدقة.\n\n"
            "للتوضيح أكثر:\n"
            "في الملخص (SUMMARY)، اشرح باختصار الخطوات التي ستتخذها لحل المشكلة.\n"
            "في الوصف (CAPTION)، صف محتويات الصورة، مع التركيز بشكل خاص على التفاصيل ذات الصلة بالسؤال.\n"
            "في التفكير (REASONING)، قدم عملية تفكير منطقية خطوة بخطوة لحل المشكلة بناءً على الصورة.\n"
            "في الخلاصة (CONCLUSION)، قدم الإجابة النهائية بتنسيق مباشر، ويجب أن تتطابق مع الإجابة الصحيحة تماماً.\n\n"
            "يجب أن يبدو التنسيق كالتالي:\n"
            "<SUMMARY>[لخص كيف ستتعامل مع المشكلة واشرح الخطوات التي ستتخذها للوصول إلى الإجابة.]</SUMMARY>"
            "<CAPTION>[قدم وصفاً تفصيلياً للصورة، مع التركيز بشكل خاص على الجوانب المتعلقة بالسؤال.]</CAPTION>"
            "<REASONING>[قدم تفسيراً منطقياً متسلسلاً للمشكلة. يجب أن يوضح هذا التفكير خطوة بخطوة.]</REASONING>"
            "<CONCLUSION>[اذكر الإجابة النهائية بتنسيق واضح ومباشر. يجب أن تتطابق مع الإجابة الصحيحة تماماً.]\n</CONCLUSION>"
            "(لا تنسَ </CONCLUSION>!)\n\n"
            "يرجى تطبيق هذا التنسيق بدقة لتحليل الصورة المعطاة والإجابة على السؤال المتعلق، مع التأكد من أن الإجابة تتطابق مع الإجابة القياسية بشكل مثالي."
        )
    })

    content.append({
        "type": "text",
        "text": "السؤال: " + question
    })

    content.append({
        "type": "text",
        "text": "الإجابة القياسية: " + standard_answer
    })

    return content

def extract_conclusion(output_text):
    """Extract the CONCLUSION section from the output"""
    if not output_text:
        return None

    # Try to extract text between <CONCLUSION> tags
    if "<CONCLUSION>" in output_text and "</CONCLUSION>" in output_text:
        start = output_text.find("<CONCLUSION>") + len("<CONCLUSION>")
        end = output_text.find("</CONCLUSION>")
        conclusion = output_text[start:end].strip()
        return conclusion

    return None

def judge_answer(standard_answer: str, conclusion: str) -> bool:
    """
    Ask the model if the conclusion is valid.
    Returns True if valid, False if invalid.
    """
    judge_content = [
        {
            "type": "text",
            "text": (
                "قيّم ما إذا كانت إجابة المساعد صالحة. أجب فقط بكلمة واحدة:\n"
                "- 'صالح' إذا كانت إجابة المساعد ليست رفضاً وتتوافق مع الإجابة القياسية في المعنى.\n"
                "- 'غير صالح' إذا كانت الإجابة رفضاً أو تختلف عن الإجابة القياسية بشكل جوهري.\n\n"
                f"الإجابة القياسية: {standard_answer}\n"
                f"إجابة المساعد: {conclusion}"
            )
        }
    ]

    judge_messages = [
        {
            "role": "user",
            "content": judge_content
        }
    ]

    max_judge_retries = 3
    for attempt in range(max_judge_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=judge_messages,
                max_tokens=10
            )
            time.sleep(1)
            judgment = response.choices[0].message.content.strip().lower()

            if "غير صالح" in judgment or "invalid" in judgment:
                return False
            if "صالح" in judgment or "valid" in judgment:
                return True
            return False

        except Exception as e:
            msg = str(e)
            if "Error code: 429" in msg and attempt < max_judge_retries - 1:
                m = re.search(r"try again in ([0-9.]+)s", msg)
                wait_s = float(m.group(1)) if m else 30.0
                wait_s = max(wait_s, 30.0)
                print(f"Rate limit in judge_answer. Waiting {wait_s:.2f}s...")
                time.sleep(wait_s)
                continue
            else:
                print(f"Error in judge_answer: {e}")
                return False

    return False

def generate_arabic_response(image, arabic_question, arabic_ground_truth, max_retries=3):
    """
    Generate response using GPT-4o with validation
    Returns: (generated_text_or_none, passed_bool)
    """
    base64_image = image_to_base64(image)

    best_attempt_text = None

    for attempt in range(max_retries):
        try:
            print(f"  Generating response (attempt {attempt + 1}/{max_retries})...")

            content = build_content(base64_image, arabic_question, arabic_ground_truth)

            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": content}],
                max_tokens=2000,
                temperature=0.7
            )
            time.sleep(1)

            result = response.choices[0].message.content.strip()
            best_attempt_text = result  # keep latest attempt
            print(f"  Response generated: {len(result)} characters")

            conclusion = extract_conclusion(result)

            if not conclusion:
                print("  ⚠ Warning: Could not extract CONCLUSION")
                continue

            print(f"  CONCLUSION: {conclusion}")
            print(f"  Ground truth: {arabic_ground_truth}")

            is_valid = judge_answer(arabic_ground_truth, conclusion)

            if is_valid:
                print("  ✓ CONCLUSION validated: صالح")
                return result, True
            else:
                print("  ✗ CONCLUSION invalid: غير صالح")
                continue

        except Exception as e:
            print(f"  Generation error: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            break

    # If never passed:
    return best_attempt_text, False

def process_dataset(dataset_name: str = "IconQA", num_samples: int = 500):
    print("=" * 60)
    print(f"Processing {dataset_name} Dataset")
    print("=" * 60)

    images_dir = f"{dataset_name.lower()}_images"
    os.makedirs(images_dir, exist_ok=True)

    output_file = f"arabic_{dataset_name.lower()}_dataset.jsonl"

    # Load already-processed IDs from existing JSONL (resume-safe)
    existing_ids = set()
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if "id" in obj:
                        existing_ids.add(str(obj["id"]))
                except json.JSONDecodeError:
                    continue
    print(f"Already processed in JSONL: {len(existing_ids)}")

    print("\nLoading dataset...")
    dataset = load_dataset("5CD-AI/LLaVA-CoT-o1-Instruct", split="train")

    # Filter for specific dataset
    filtered_samples = []
    for sample in dataset:
        if identify_dataset_source(sample) == dataset_name:
            filtered_samples.append(sample)
            if len(filtered_samples) >= num_samples:
                break

    print(f"Found {len(filtered_samples)} {dataset_name} samples")

    results = []

    for idx, sample in enumerate(tqdm(filtered_samples, desc=f"Processing {dataset_name}")):
        image_path = None
        try:
            #  Stable id for skipping + saving
            sample_id = str(sample.get("id", f"{dataset_name.lower()}_{idx}"))

            #  Skip if already exists
            if sample_id in existing_ids:
                print(f"⏭️ Skip existing id: {sample_id}")
                continue

            image = sample.get("image")

            # Get question
            question = ""
            if "conversations" in sample and len(sample["conversations"]) > 0:
                question = sample["conversations"][0].get("value", "")
            elif "question" in sample:
                question = sample["question"]

            # Get ground truth
            ground_truth = None
            if "conversations" in sample and len(sample["conversations"]) > 1:
                ground_truth = sample["conversations"][1].get("value", "")
            elif "answer" in sample:
                ground_truth = sample["answer"]
            elif "ground_truth" in sample:
                ground_truth = sample["ground_truth"]

            question_clean = question.replace("<image>", "").replace("<img>", "").strip()

            print(f"\n{'='*60}")
            print(f"Processing sample {idx+1}/{len(filtered_samples)}")
            print(f"ID: {sample_id}")
            print(f"Original question: {question_clean[:100]}")
            print(f"Ground truth: {ground_truth}")
            print(f"{'='*60}")

            if not question_clean:
                print("✗ Empty question, skipping")
                continue

            #  Save image first (so we can delete it if fail)
            image_path = os.path.join(images_dir, f"{dataset_name.lower()}_image_{1847+idx}.png")
            image.save(image_path)
            print(f"✓ Image saved: {image_path}")

            # Translate question
            print("Step 1: Translating question to Arabic...")
            arabic_question = translate_text(question_clean, "question")
            print(f"✓ Arabic question: {arabic_question[:80]}...")

            # Translate ground truth
            print("Step 2: Translating ground truth to Arabic...")
            arabic_ground_truth = translate_text(str(ground_truth), "answer")
            print(f"✓ Arabic ground truth: {arabic_ground_truth}")

            # Generate response + validation
            print("Step 3: Generating response with GPT-4o...")
            arabic_output, passed = generate_arabic_response(
                image,
                arabic_question,
                arabic_ground_truth
            )

            if (not passed) or (not arabic_output):
                print("✗ Sample failed validation. Deleting image and skipping saving row.")
                try:
                    if image_path and os.path.exists(image_path):
                        os.remove(image_path)
                        print(f"🗑️ Deleted image: {image_path}")
                except Exception as del_err:
                    print(f"⚠ Could not delete image: {del_err}")
                continue  # ✅ do not save JSONL row

            #  Store result ONLY if passed
            result = {
                "id": sample_id,           # ✅ stable id
                "image": image_path,
                "question": arabic_question,
                "ground_truth": arabic_ground_truth,
                "augmented_answer": arabic_output
            }

            results.append(result)

            # Save incrementally
            with open(output_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            #  Update existing ids so duplicates in same run skip too
            existing_ids.add(sample_id)

            print(f"✓ Sample {idx+1} completed and saved")

        except Exception as e:
            print(f"✗ Error processing sample {idx}: {e}")
            import traceback
            traceback.print_exc()

            # Delete image if it was created
            try:
                if image_path and os.path.exists(image_path):
                    os.remove(image_path)
                    print(f"🗑️ Deleted image due to error: {image_path}")
            except Exception:
                pass

            continue

    # Save complete results
    with open(f"arabic_{dataset_name.lower()}_complete.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 60)
    print(f"✓ Processing complete! Generated {len(results)} NEW samples.")
    print("=" * 60)
    print(f"Output JSONL: {output_file}")
    print(f"Images folder: {images_dir}")


if __name__ == "__main__":
    # Process IconQA dataset
    process_dataset(dataset_name="IconQA", num_samples=4000)

Processing IconQA Dataset
Already processed in JSONL: 1541

Loading dataset...


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Found 4000 IconQA samples


Processing IconQA:   0%|          | 0/4000 [00:00<?, ?it/s]

...

  ✓ CONCLUSION validated: صالح
✓ Sample 1846 completed and saved

Processing sample 1847/4000
ID: iconqa/iconqa_data/iconqa/train/choose_txt/80773/image.png
Original question: Please answer the question below, explaining your reasoning step by step before providing the final 
Ground truth: B
✓ Image saved: iconqa_images/iconqa_image_1846.png
Step 1: Translating question to Arabic...
✓ Arabic question: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة ...
Step 2: Translating ground truth to Arabic...
✓ Arabic ground truth: ب
Step 3: Generating response with GPT-4o...
  Generating response (attempt 1/3)...
  Response generated: 508 characters
  CONCLUSION: ب
  Ground truth: ب


Processing IconQA:  46%|████▌     | 1847/4000 [1:34:29<6:39:03, 11.12s/it]

  ✓ CONCLUSION validated: صالح
✓ Sample 1847 completed and saved

Processing sample 1848/4000
ID: iconqa/iconqa_data/iconqa/train/choose_txt/41229/image.png
Original question: Please answer the question below, explaining your reasoning step by step before providing the final 
Ground truth: A
✓ Image saved: iconqa_images/iconqa_image_1847.png
Step 1: Translating question to Arabic...
✓ Arabic question: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة ...
Step 2: Translating ground truth to Arabic...
✓ Arabic ground truth: نعم.
Step 3: Generating response with GPT-4o...
  Generating response (attempt 1/3)...
  Response generated: 606 characters
  CONCLUSION: [الإجابة هي: أ. حوالي 20]
  Ground truth: نعم.
  ✗ CONCLUSION invalid: غير صالح
  Generating response (attempt 2/3)...


Processing IconQA:  46%|████▌     | 1847/4000 [1:34:44<1:50:25,  3.08s/it]


KeyboardInterrupt: 

In [ ]:
from google.colab import files
!zip -r iconqa_images.zip iconqa_images
files.download("iconqa_images.zip")
files.download("arabic_iconqa_dataset.jsonl")

updating: iconqa_images/ (stored 0%)
updating: iconqa_images/iconqa_image_641.png (deflated 12%)
updating: iconqa_images/iconqa_image_584.png (stored 0%)
updating: iconqa_images/iconqa_image_1529.png (deflated 2%)
updating: iconqa_images/iconqa_image_1227.png (deflated 4%)
updating: iconqa_images/iconqa_image_1623.png (deflated 5%)
updating: iconqa_images/iconqa_image_1451.png (deflated 2%)
updating: iconqa_images/iconqa_image_404.png (deflated 4%)
updating: iconqa_images/iconqa_image_214.png (deflated 72%)
updating: iconqa_images/iconqa_image_1615.png (deflated 73%)
updating: iconqa_images/iconqa_image_751.png (deflated 1%)
updating: iconqa_images/iconqa_image_679.png (deflated 0%)
updating: iconqa_images/iconqa_image_632.png (stored 0%)
updating: iconqa_images/iconqa_image_288.png (deflated 6%)
updating: iconqa_images/iconqa_image_1458.png (stored 0%)
updating: iconqa_images/iconqa_image_1311.png (deflated 1%)
updating: iconqa_images/iconqa_image_133.png (deflated 11%)
updating: icon

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import json
import time
import re
from typing import Dict
from datasets import load_dataset
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from google.colab import userdata

# Initialize OpenAI client
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

def load_existing_ids(output_file: str) -> set:
    """Read JSONL and return set of existing 'id' values."""
    existing = set()
    if not os.path.exists(output_file):
        return existing

    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if "id" in obj:
                    existing.add(str(obj["id"]))
            except json.JSONDecodeError:
                continue
    return existing


def get_sample_id(sample, dataset_name: str, idx: int) -> str:
    """Stable id for skipping."""
    return str(sample.get("id", f"{dataset_name.lower()}_{idx}"))

def image_to_base64(image):
    """Convert PIL Image to base64 string"""
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode()

def identify_dataset_source(entry: Dict) -> str:
    """
    Identify which dataset an entry belongs to

    Args:
        entry: Dataset entry

    Returns:
        Dataset name or None
    """
    # Check if entry has 'id' field with dataset path
    entry_id = entry.get('id', '')

    # Handle image field - might be PIL Image or string
    image_path = entry.get('image', '')
    if hasattr(image_path, '__class__') and 'Image' in str(type(image_path)):
        # It's a PIL Image, can't search in it, use empty string
        image_path = ''
    else:
        image_path = str(image_path).lower()

    # Common patterns in LLaVA-CoT-o1-Instruct

    # CLEVER: has 'clever'
    if 'clever' in entry_id.lower() or 'clever' in image_path.lower():
        return 'CLEVER'

    return None

def translate_text(text: str, text_type: str = "question") -> str:
    """
    Translate text to Arabic

    Args:
        text: English text
        text_type: Type of text ("question" or "answer")

    Returns:
        Arabic translation
    """
    if text_type == "question":
        prompt = f"""ترجم السؤال التالي إلى اللغة العربية الفصحى:

{text}

قواعد:
- ترجم السؤال الكامل بما في ذلك أي مقدمات أو تعليمات
- استخدم لغة علمية دقيقة
- لا تضف أي تفسيرات إضافية

الترجمة:"""
    else:  # answer
        prompt = f"""ترجم الإجابة التالية إلى اللغة العربية الفصحى:

{text}

قواعد:
- ترجم الإجابة فقط دون إضافات
- استخدم لغة دقيقة

الترجمة:"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": "أنت مترجم محترف متخصص في ترجمة النصوص العلمية."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_tokens=500
        )
        translation = response.choices[0].message.content.strip()
        time.sleep(1)

        return translation

    except Exception as e:
        print(f"Translation error: {e}")
        return text

def build_content(base64_image, question, standard_answer):
    """Build the multimodal content for the chat completion."""
    content = []

    if base64_image is not None:
        content.append({
            "type": "image_url",
            "image_url": {"url": "data:image/jpeg;base64," + base64_image}
        })

    content.append({
        "type": "text",
        "text": (
            "لدي صورة وسؤال أريد منك الإجابة عليه. أحتاج منك اتباع التنسيق بدقة مع أربعة أقسام محددة: الملخص (SUMMARY)، الوصف (CAPTION)، التفكير (REASONING)، والخلاصة (CONCLUSION). "
            "من الضروري جداً الالتزام بهذا الهيكل تماماً وأن تتطابق الإجابة النهائية في الخلاصة مع الإجابة الصحيحة القياسية بدقة.\n\n"
            "للتوضيح أكثر:\n"
            "في الملخص (SUMMARY)، اشرح باختصار الخطوات التي ستتخذها لحل المشكلة.\n"
            "في الوصف (CAPTION)، صف محتويات الصورة، مع التركيز بشكل خاص على التفاصيل ذات الصلة بالسؤال.\n"
            "في التفكير (REASONING)، قدم عملية تفكير منطقية خطوة بخطوة لحل المشكلة بناءً على الصورة.\n"
            "في الخلاصة (CONCLUSION)، قدم الإجابة النهائية بتنسيق مباشر، ويجب أن تتطابق مع الإجابة الصحيحة تماماً.\n\n"
            "يجب أن يبدو التنسيق كالتالي:\n"
            "<SUMMARY>[لخص كيف ستتعامل مع المشكلة واشرح الخطوات التي ستتخذها للوصول إلى الإجابة.]</SUMMARY>"
            "<CAPTION>[قدم وصفاً تفصيلياً للصورة، مع التركيز بشكل خاص على الجوانب المتعلقة بالسؤال.]</CAPTION>"
            "<REASONING>[قدم تفسيراً منطقياً متسلسلاً للمشكلة. يجب أن يوضح هذا التفكير خطوة بخطوة.]</REASONING>"
            "<CONCLUSION>[اذكر الإجابة النهائية بتنسيق واضح ومباشر. يجب أن تتطابق مع الإجابة الصحيحة تماماً.]\n</CONCLUSION>"
            "(لا تنسَ </CONCLUSION>!)\n\n"
            "يرجى تطبيق هذا التنسيق بدقة لتحليل الصورة المعطاة والإجابة على السؤال المتعلق، مع التأكد من أن الإجابة تتطابق مع الإجابة القياسية بشكل مثالي."
        )
    })

    content.append({
        "type": "text",
        "text": "السؤال: " + question
    })

    content.append({
        "type": "text",
        "text": "الإجابة القياسية: " + standard_answer
    })

    return content

def extract_conclusion(output_text):
    """Extract the CONCLUSION section from the output"""
    if not output_text:
        return None

    # Try to extract text between <CONCLUSION> tags
    if "<CONCLUSION>" in output_text and "</CONCLUSION>" in output_text:
        start = output_text.find("<CONCLUSION>") + len("<CONCLUSION>")
        end = output_text.find("</CONCLUSION>")
        conclusion = output_text[start:end].strip()
        return conclusion

    return None

def judge_answer(standard_answer: str, conclusion: str) -> bool:
    """
    Ask the model if the conclusion is valid.
    Returns True if valid, False if invalid.
    """
    judge_content = [
        {
            "type": "text",
            "text": (
                "قيّم ما إذا كانت إجابة المساعد صالحة. أجب فقط بكلمة واحدة:\n"
                "- 'صالح' إذا كانت إجابة المساعد ليست رفضاً وتتوافق مع الإجابة القياسية في المعنى.\n"
                "- 'غير صالح' إذا كانت الإجابة رفضاً أو تختلف عن الإجابة القياسية بشكل جوهري.\n\n"
                f"الإجابة القياسية: {standard_answer}\n"
                f"إجابة المساعد: {conclusion}"
            )
        }
    ]

    judge_messages = [
        {
            "role": "user",
            "content": judge_content
        }
    ]

    max_judge_retries = 3
    for attempt in range(max_judge_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=judge_messages,
                max_tokens=10
            )
            time.sleep(1)
            judgment = response.choices[0].message.content.strip().lower()

            if "غير صالح" in judgment or "invalid" in judgment:
                return False
            if "صالح" in judgment or "valid" in judgment:
                return True
            return False

        except Exception as e:
            msg = str(e)
            if "Error code: 429" in msg and attempt < max_judge_retries - 1:
                m = re.search(r"try again in ([0-9.]+)s", msg)
                wait_s = float(m.group(1)) if m else 30.0
                wait_s = max(wait_s, 30.0)
                print(f"Rate limit in judge_answer. Waiting {wait_s:.2f}s...")
                time.sleep(wait_s)
                continue
            else:
                print(f"Error in judge_answer: {e}")
                return False

    return False

def generate_arabic_response(image, arabic_question, arabic_ground_truth, max_retries=3):
    """
    Generate response using GPT-4o with validation
    Returns: (generated_text_or_none, passed_bool)
    """
    base64_image = image_to_base64(image)

    best_attempt_text = None

    for attempt in range(max_retries):
        try:
            print(f"  Generating response (attempt {attempt + 1}/{max_retries})...")

            content = build_content(base64_image, arabic_question, arabic_ground_truth)

            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": content}],
                max_tokens=2000,
                temperature=0.7
            )
            time.sleep(1)

            result = response.choices[0].message.content.strip()
            best_attempt_text = result  # keep latest attempt
            print(f"  Response generated: {len(result)} characters")

            conclusion = extract_conclusion(result)

            if not conclusion:
                print("  ⚠ Warning: Could not extract CONCLUSION")
                continue

            print(f"  CONCLUSION: {conclusion}")
            print(f"  Ground truth: {arabic_ground_truth}")

            is_valid = judge_answer(arabic_ground_truth, conclusion)

            if is_valid:
                print("  ✓ CONCLUSION validated: صالح")
                return result, True
            else:
                print("  ✗ CONCLUSION invalid: غير صالح")
                continue

        except Exception as e:
            print(f"  Generation error: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            break

    # If never passed:
    return best_attempt_text, False

def process_dataset(dataset_name: str = "IconQA", num_samples: int = 500):
    print("=" * 60)
    print(f"Processing {dataset_name} Dataset")
    print("=" * 60)

    images_dir = "CLEVER"
    os.makedirs(images_dir, exist_ok=True)

    output_file = f"arabic_{dataset_name.lower()}_dataset.jsonl"

    # ✅ Load already-processed IDs from existing JSONL (resume-safe)
    existing_ids = set()
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if "id" in obj:
                        existing_ids.add(str(obj["id"]))
                except json.JSONDecodeError:
                    continue
    print(f"Already processed in JSONL: {len(existing_ids)}")

    print("\nLoading dataset...")
    dataset = load_dataset("5CD-AI/LLaVA-CoT-o1-Instruct", split="train")

    # Filter for specific dataset
    filtered_samples = []
    for sample in dataset:
        if identify_dataset_source(sample) == dataset_name:
            filtered_samples.append(sample)
            if len(filtered_samples) >= num_samples:
                break

    print(f"Found {len(filtered_samples)} {dataset_name} samples")

    results = []

    for idx, sample in enumerate(tqdm(filtered_samples, desc=f"Processing {dataset_name}")):
        image_path = None
        try:
            #  Stable id for skipping + saving
            sample_id = str(sample.get("id", f"{dataset_name.lower()}_{idx}"))

            # Skip if already exists
            if sample_id in existing_ids:
                print(f"⏭️ Skip existing id: {sample_id}")
                continue

            image = sample.get("image")

            # Get question
            question = ""
            if "conversations" in sample and len(sample["conversations"]) > 0:
                question = sample["conversations"][0].get("value", "")
            elif "question" in sample:
                question = sample["question"]

            # Get ground truth
            ground_truth = None
            if "conversations" in sample and len(sample["conversations"]) > 1:
                ground_truth = sample["conversations"][1].get("value", "")
            elif "answer" in sample:
                ground_truth = sample["answer"]
            elif "ground_truth" in sample:
                ground_truth = sample["ground_truth"]

            question_clean = question.replace("<image>", "").replace("<img>", "").strip()

            print(f"\n{'='*60}")
            print(f"Processing sample {idx+1}/{len(filtered_samples)}")
            print(f"ID: {sample_id}")
            print(f"Original question: {question_clean[:100]}")
            print(f"Ground truth: {ground_truth}")
            print(f"{'='*60}")

            if not question_clean:
                print("✗ Empty question, skipping")
                continue

            #  Save image first
            image_path = os.path.join(images_dir, f"{dataset_name.lower()}_image_{idx}.png")
            image.save(image_path)
            print(f"✓ Image saved: {image_path}")

            # Translate question
            print("Step 1: Translating question to Arabic...")
            arabic_question = translate_text(question_clean, "question")
            print(f"✓ Arabic question: {arabic_question[:80]}...")


            # Generate response + validation
            print("Step 3: Generating response with GPT-4o...")
            arabic_output, passed = generate_arabic_response(
                image,
                arabic_question,
                ground_truth
            )

            if (not passed) or (not arabic_output):
                print("✗ Sample failed validation. Deleting image and skipping saving row.")
                try:
                    if image_path and os.path.exists(image_path):
                        os.remove(image_path)
                        print(f"🗑️ Deleted image: {image_path}")
                except Exception as del_err:
                    print(f"⚠ Could not delete image: {del_err}")
                continue  # ✅ do not save JSONL row

            # Store result ONLY if passed
            result = {
                "id": sample_id,           # ✅ stable id
                "image": image_path,
                "question": arabic_question,
                "ground_truth": ground_truth,
                "augmented_answer": arabic_output
            }

            results.append(result)

            # Save incrementally
            with open(output_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            # Update existing ids so duplicates in same run skip too
            existing_ids.add(sample_id)

            print(f"✓ Sample {idx+1} completed and saved")

        except Exception as e:
            print(f"✗ Error processing sample {idx}: {e}")
            import traceback
            traceback.print_exc()

            # Delete image if it was created
            try:
                if image_path and os.path.exists(image_path):
                    os.remove(image_path)
                    print(f"🗑️ Deleted image due to error: {image_path}")
            except Exception:
                pass

            continue

    # Save complete results
    with open(f"arabic_{dataset_name.lower()}_complete.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 60)
    print(f"✓ Processing complete! Generated {len(results)} NEW samples.")
    print("=" * 60)
    print(f"Output JSONL: {output_file}")
    print(f"Images folder: {images_dir}")


if __name__ == "__main__":
    # Process IconQA dataset
    process_dataset(dataset_name="CLEVER", num_samples=2666)

Processing IconQA Dataset
Already processed in JSONL: 1541

Loading dataset...


Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Found 4000 IconQA samples


Processing IconQA:   0%|          | 0/4000 [00:00<?, ?it/s]

...

  ✓ CONCLUSION validated: صالح
✓ Sample 1846 completed and saved

Processing sample 1847/4000
ID: iconqa/iconqa_data/iconqa/train/choose_txt/80773/image.png
Original question: Please answer the question below, explaining your reasoning step by step before providing the final 
Ground truth: B
✓ Image saved: iconqa_images/iconqa_image_1846.png
Step 1: Translating question to Arabic...
✓ Arabic question: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة ...
Step 2: Translating ground truth to Arabic...
✓ Arabic ground truth: ب
Step 3: Generating response with GPT-4o...
  Generating response (attempt 1/3)...
  Response generated: 508 characters
  CONCLUSION: ب
  Ground truth: ب


Processing IconQA:  46%|████▌     | 1847/4000 [1:34:29<6:39:03, 11.12s/it]

  ✓ CONCLUSION validated: صالح
✓ Sample 1847 completed and saved

Processing sample 1848/4000
ID: iconqa/iconqa_data/iconqa/train/choose_txt/41229/image.png
Original question: Please answer the question below, explaining your reasoning step by step before providing the final 
Ground truth: A
✓ Image saved: iconqa_images/iconqa_image_1847.png
Step 1: Translating question to Arabic...
✓ Arabic question: يرجى الإجابة على السؤال أدناه، موضحًا خطوات تفكيرك خطوة بخطوة قبل تقديم الإجابة ...
Step 2: Translating ground truth to Arabic...
✓ Arabic ground truth: نعم.
Step 3: Generating response with GPT-4o...
  Generating response (attempt 1/3)...
  Response generated: 606 characters
  CONCLUSION: [الإجابة هي: أ. حوالي 20]
  Ground truth: نعم.
  ✗ CONCLUSION invalid: غير صالح
  Generating response (attempt 2/3)...


Processing IconQA:  46%|████▌     | 1847/4000 [1:34:44<1:50:25,  3.08s/it]


KeyboardInterrupt: 

In [ ]:
import os
import json
import time
import re
import base64
from io import BytesIO
from google.colab import userdata

from PIL import Image
from tqdm import tqdm
from openai import OpenAI

client = OpenAI()

INPUT_JSONL  = "arabic_dvqa_dataset.jsonl"
OUTPUT_JSONL = "output.jsonl"

GEN_MODEL   = "gpt-4o"
JUDGE_MODEL = "gpt-4o"

MAX_RETRIES = 3
SLEEP_BETWEEN_CALLS = 0.8
MAX_TOKENS_GEN = 1400


def image_to_base64(img: Image.Image) -> str:
    buf = BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=95)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def build_content(b64: str, question: str, standard_answer: str):
    return [
        {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + b64}},
        {"type": "text", "text": (
            "لدي صورة وسؤال أريد منك الإجابة عليه. أحتاج منك اتباع التنسيق بدقة مع أربعة أقسام محددة: الملخص (SUMMARY)، الوصف (CAPTION)، التفكير (REASONING)، والخلاصة (CONCLUSION). "
            "من الضروري جداً الالتزام بهذا الهيكل تماماً وأن تتطابق الإجابة النهائية في الخلاصة مع الإجابة الصحيحة القياسية بدقة.\n\n"
            "للتوضيح أكثر:\n"
            "في الملخص (SUMMARY)، اشرح باختصار الخطوات التي ستتخذها لحل المشكلة.\n"
            "في الوصف (CAPTION)، صف محتويات الصورة، مع التركيز بشكل خاص على التفاصيل ذات الصلة بالسؤال.\n"
            "في التفكير (REASONING)، قدم عملية تفكير منطقية خطوة بخطوة لحل المشكلة بناءً على الصورة.\n"
            "في الخلاصة (CONCLUSION)، قدم الإجابة النهائية بتنسيق مباشر، ويجب أن تتطابق مع الإجابة الصحيحة تماماً.\n\n"
            "يجب أن يبدو التنسيق كالتالي:\n"
            "<SUMMARY>[لخص كيف ستتعامل مع المشكلة واشرح الخطوات التي ستتخذها للوصول إلى الإجابة.]</SUMMARY>"
            "<CAPTION>[قدم وصفاً تفصيلياً للصورة، مع التركيز بشكل خاص على الجوانب المتعلقة بالسؤال.]</CAPTION>"
            "<REASONING>[قدم تفسيراً منطقياً متسلسلاً للمشكلة. يجب أن يوضح هذا التفكير خطوة بخطوة.]</REASONING>"
            "<CONCLUSION>[اذكر الإجابة النهائية بتنسيق واضح ومباشر. يجب أن تتطابق مع الإجابة الصحيحة تماماً.]\n</CONCLUSION>"
            "(لا تنسَ </CONCLUSION>!)\n\n"
            "يرجى تطبيق هذا التنسيق بدقة لتحليل الصورة المعطاة والإجابة على السؤال المتعلق، مع التأكد من أن الإجابة تتطابق مع الإجابة القياسية بشكل مثالي."
        )},
        {"type": "text", "text": "السؤال: " + question},
        {"type": "text", "text": "الإجابة القياسية: " + str(standard_answer)},
    ]

def extract_conclusion(text: str):
    if not text:
        return None
    m = re.search(r"<CONCLUSION>(.*?)</CONCLUSION>", text, flags=re.DOTALL)
    return m.group(1).strip() if m else None

def judge_answer(standard: str, conclusion: str) -> bool:
    prompt = (
        "هل إجابة المساعد تطابق الإجابة القياسية (نصياً أو بالمعنى)؟\n"
        "أجب بكلمة واحدة فقط: صالح أو غير صالح.\n\n"
        f"الإجابة القياسية: {standard}\n"
        f"إجابة المساعد: {conclusion}"
    )
    try:
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=10
        )
        out = (resp.choices[0].message.content or "").strip()
        return ("صالح" in out) and ("غير" not in out)
    except Exception as e:
        print(f"⚠ judge error: {e}")
        return False

def generate_augmented(img: Image.Image, q: str, gt: str):
    b64 = image_to_base64(img)
    best = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=GEN_MODEL,
                messages=[{"role": "user", "content": build_content(b64, q, gt)}],
                temperature=0.7,
                max_tokens=MAX_TOKENS_GEN
            )
            text = (resp.choices[0].message.content or "").strip()
            best = text

            concl = extract_conclusion(text)
            if not concl:
                time.sleep(SLEEP_BETWEEN_CALLS)
                continue

            if judge_answer(str(gt), concl):
                return text, True

            time.sleep(SLEEP_BETWEEN_CALLS)

        except Exception as e:
            print(f"⚠ generation error (attempt {attempt}): {e}")
            time.sleep(SLEEP_BETWEEN_CALLS)

    return best, False

def load_existing_ids(path: str) -> set:
    ids = set()
    if not os.path.exists(path):
        return ids
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if "id" in obj:
                    ids.add(str(obj["id"]))
            except:
                pass
    return ids

def main():

    with open(INPUT_JSONL, "r", encoding="utf-8") as fin, \
         open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:

        for line in tqdm(fin, desc="Regenerating"):
            if not line.strip():
                continue

            obj = json.loads(line)

            sid = str(obj.get("id", "")).strip()
            img_path = obj.get("image")
            q = (obj.get("question") or "").strip()
            gt = obj.get("ground_truth")

            if not img_path or not os.path.exists(img_path):
                continue
            if not q or gt is None:
                continue

            img = Image.open(img_path)

            augmented, passed = generate_augmented(img, q, str(gt))

            if not passed or not augmented:
                continue

            out_row = {
                "id": sid,
                "image": img_path,
                "question": q,
                "ground_truth": gt,
                "augmented_answer": augmented
            }

            fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")

    print("✅ Done.")
    print("New clean file:", OUTPUT_JSONL)

if __name__ == "__main__":
    main()